In [2]:
import numpy as np
import math
import time
import warnings
warnings.filterwarnings('ignore')
from itertools import permutations

from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from scipy.optimize import minimize


# ── Utility Functions ──

def euclidean_distance(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def build_distance_matrix(nodes):
    n = len(nodes)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i,j] = euclidean_distance(nodes[i], nodes[j])
    return D


# ── Phase 1: Sweep Algorithm (Classical Clustering) ──

def sweep_decomposition(nodes, Nv, C):
    """Cluster customers by polar angle from depot, respecting capacity C."""
    depot = nodes[0]
    customers = []
    for i in range(1, len(nodes)):
        angle = math.atan2(nodes[i][1] - depot[1], nodes[i][0] - depot[0])
        customers.append((i, angle))
    customers.sort(key=lambda x: x[1])

    clusters = []
    for start in range(0, len(customers), C):
        if len(clusters) < Nv:
            chunk = customers[start : start + C]
            clusters.append([c[0] for c in chunk])
    return clusters


# ── Phase 2: TSP via QAOA (Quantum Routing) ──

def build_tsp_qubo(cluster_indices, dist_matrix):
    """Build upper-triangular QUBO for TSP (position formulation).
    Variables x[i,s] = 1 if cluster node i is at route position s.
    n customers → n² binary variables / qubits."""
    n = len(cluster_indices)
    N = n * n
    Q = np.zeros((N, N))
    idx = lambda i, s: i * n + s

    # Penalty coefficient (must dominate route costs)
    max_d = max(dist_matrix[a, b]
                for a in [0] + cluster_indices
                for b in [0] + cluster_indices if a != b)
    A = max_d * n * 1.5

    # Constraint: each customer visited exactly once  Σ_s x[i,s] = 1
    for i in range(n):
        for s in range(n):
            Q[idx(i,s), idx(i,s)] -= A
        for s in range(n):
            for t in range(s+1, n):
                Q[idx(i,s), idx(i,t)] += 2 * A

    # Constraint: each position filled by exactly one customer  Σ_i x[i,s] = 1
    for s in range(n):
        for i in range(n):
            Q[idx(i,s), idx(i,s)] -= A
        for i in range(n):
            for j in range(i+1, n):
                Q[idx(i,s), idx(j,s)] += 2 * A

    # Objective: route distance (depot → pos 0, transitions, pos n-1 → depot)
    for i, ci in enumerate(cluster_indices):
        Q[idx(i,0), idx(i,0)]     += dist_matrix[0, ci]   # depot → first
        Q[idx(i,n-1), idx(i,n-1)] += dist_matrix[ci, 0]   # last → depot
        for j, cj in enumerate(cluster_indices):
            if i != j:
                for s in range(n - 1):
                    a, b = idx(i, s), idx(j, s+1)
                    if a <= b:
                        Q[a, b] += dist_matrix[ci, cj]
                    else:
                        Q[b, a] += dist_matrix[ci, cj]
    return Q


def qubo_to_ising(Q):
    """Convert upper-triangular QUBO matrix to a SparsePauliOp Ising Hamiltonian.
    x_k = (1 - Z_k) / 2"""
    n = Q.shape[0]
    constant = 0.0
    h = np.zeros(n)
    J = {}

    for k in range(n):
        constant += Q[k, k] / 2.0
        h[k]     -= Q[k, k] / 2.0

    for k in range(n):
        for l in range(k+1, n):
            if abs(Q[k, l]) > 1e-12:
                constant += Q[k, l] / 4.0
                h[k]     -= Q[k, l] / 4.0
                h[l]     -= Q[k, l] / 4.0
                J[(k, l)] = Q[k, l] / 4.0

    # Build Pauli list  (qiskit little-endian: rightmost char = qubit 0)
    pauli_list = []
    for k in range(n):
        if abs(h[k]) > 1e-12:
            lbl = ['I'] * n
            lbl[n - 1 - k] = 'Z'
            pauli_list.append((''.join(lbl), h[k]))

    for (k, l), coef in J.items():
        if abs(coef) > 1e-12:
            lbl = ['I'] * n
            lbl[n - 1 - k] = 'Z'
            lbl[n - 1 - l] = 'Z'
            pauli_list.append((''.join(lbl), coef))

    if not pauli_list:
        pauli_list = [('I' * n, 0.0)]

    return SparsePauliOp.from_list(pauli_list).simplify(), constant


def decode_bitstring(bitstring, n, cluster_indices, dist_matrix):
    """Decode a measurement bitstring → (ordered_customer_ids, route_distance).
    Returns (None, inf) when the bitstring violates a constraint."""
    bits = [int(b) for b in reversed(bitstring)]
    while len(bits) < n * n:
        bits.append(0)

    x = np.zeros((n, n), dtype=int)
    for i in range(n):
        for s in range(n):
            x[i, s] = bits[i * n + s]

    if not all(x[i, :].sum() == 1 for i in range(n)):
        return None, float('inf')
    if not all(x[:, s].sum() == 1 for s in range(n)):
        return None, float('inf')

    order = [0] * n
    for i in range(n):
        order[int(np.argmax(x[i, :]))] = cluster_indices[i]

    d = dist_matrix[0, order[0]]
    for k in range(n - 1):
        d += dist_matrix[order[k], order[k+1]]
    d += dist_matrix[order[-1], 0]
    return order, d


def solve_tsp_qaoa(cluster_indices, dist_matrix, reps=2, shots=4096, restarts=5):
    """Solve TSP on one cluster via QAOA.  Returns (route, distance, n_qubits, n_gates, exec_time)."""
    n = len(cluster_indices)

    # Trivial single-customer cluster
    if n == 1:
        d = dist_matrix[0, cluster_indices[0]] * 2
        return list(cluster_indices), d, 0, 0, 0.0

    t0 = time.time()

    # QUBO → Ising
    Q = build_tsp_qubo(cluster_indices, dist_matrix)
    H, offset = qubo_to_ising(Q)

    # QAOA ansatz
    ansatz = QAOAAnsatz(H, reps=reps)
    estimator = StatevectorEstimator()

    def cost_func(params):
        return estimator.run([(ansatz, H, params)]).result()[0].data.evs

    # Optimize with random restarts
    best_params, best_energy = None, float('inf')
    for _ in range(restarts):
        x0 = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)
        res = minimize(cost_func, x0, method='COBYLA', options={'maxiter': 300})
        if res.fun < best_energy:
            best_energy = res.fun
            best_params = res.x

    # Sample the optimized circuit
    ansatz_meas = ansatz.measure_all(inplace=False)
    bound = ansatz_meas.assign_parameters(best_params)
    sampler = StatevectorSampler()
    counts = sampler.run([bound], shots=shots).result()[0].data.meas.get_counts()

    # Pick the best valid route from samples
    best_route, best_dist = None, float('inf')
    for bs in counts:
        route, d = decode_bitstring(bs, n, cluster_indices, dist_matrix)
        if route is not None and d < best_dist:
            best_dist = d
            best_route = route

    # Brute-force fallback (safe for small n)
    if best_route is None:
        for perm in permutations(cluster_indices):
            d = dist_matrix[0, perm[0]]
            for k in range(len(perm) - 1):
                d += dist_matrix[perm[k], perm[k+1]]
            d += dist_matrix[perm[-1], 0]
            if d < best_dist:
                best_dist = d
                best_route = list(perm)

    elapsed = time.time() - t0
    decomposed = ansatz.decompose()
    n_qubits = ansatz.num_qubits
    n_gates = sum(decomposed.count_ops().values())
    return best_route, best_dist, n_qubits, n_gates, elapsed


# ── Instance Library ──

instances = {
    1: {"Nv": 2, "C": 5, "nodes": [(0,0), (-2,2), (-5,8), (2,3)]},
    2: {"Nv": 2, "C": 2, "nodes": [(0,0), (-2,2), (-5,8), (2,3)]},
    3: {"Nv": 3, "C": 2, "nodes": [(0,0), (-2,2), (-5,8), (2,3), (5,7), (2,4), (2,-3)]},
    4: {"Nv": 4, "C": 3, "nodes": [(0,0), (-2,2), (-5,8), (6,3), (4,4), (3,2),
                                     (0,2), (-2,3), (-4,3), (2,3), (2,7), (-2,5), (-1,4)]},
}

In [5]:
# --- SET YOUR INSTANCE HERE ---
selected_id = 2

# Load the data
config = instances[selected_id]
Nv = config["Nv"]
C  = config["C"]
nodes = config["nodes"]

print(f"Loaded Instance {selected_id}: {len(nodes)-1} customers, {Nv} vehicles, Capacity {C}")

Loaded Instance 2: 3 customers, 2 vehicles, Capacity 2


In [6]:
# 1. Distance matrix
D = build_distance_matrix(nodes)

# 2. Classical Decomposition (Sweep)
clusters = sweep_decomposition(nodes, Nv, C)
print(f"Sweep clusters: {clusters}\n")

# 3. Quantum Routing (QAOA per cluster)
total_distance = 0
routes = []
max_qubits = 0
total_gates = 0
total_time  = 0

for i, cluster in enumerate(clusters):
    route, dist, nq, ng, et = solve_tsp_qaoa(cluster, D, reps=2)
    full_route = [0] + route + [0]
    routes.append(full_route)
    total_distance += dist
    max_qubits = max(max_qubits, nq)
    total_gates += ng
    total_time  += et
    print(f"  r{i+1}: {', '.join(map(str, full_route))}  |  Distance: {dist:.2f}  |  Qubits: {nq}")

print(f"\n{'='*40}")
print(f"Instance {selected_id}  —  Total Distance: {total_distance:.2f}")
print(f"Max Qubits: {max_qubits}  |  Total Gates: {total_gates}  |  Time: {total_time:.2f}s")

Sweep clusters: [[3, 2], [1]]

  r1: 0, 2, 3, 0  |  Distance: 21.64  |  Qubits: 4
  r2: 0, 1, 0  |  Distance: 5.66  |  Qubits: 0

Instance 2  —  Total Distance: 27.30
Max Qubits: 4  |  Total Gates: 8  |  Time: 85.12s
